# Stage 04 — Calibration

Derives and applies gas-concentration calibration for the four WYO-platform gas
analyzers (**Picarro, Ultra460, Ultra321, Pico017**), using the tank/dilution
sequences logged in `raw/calibration/tank_details.txt`.

All the reusable machinery — manifest parsing, both fitting methods, applying
coefficients, and the plotting primitives — lives in **`src/calibration.py`** (imported
as `cal` below). This notebook is the *narrated orchestration*: campaign-specific
configuration, step-by-step explanation, and the plots. No fitting or plotting logic is
defined inline here.

### Two independent methods, kept clearly separate

1. **CH4 & C3H8 — tank-anchored, multi-point (Section C).** Every instrument (including
   Picarro — nothing is assumed exempt) is regressed against known tank concentrations.
   **Feb 12** is the canonical event: the only one spanning the full 0–57 ppm dilution
   ladder (all 5 dilutions + NOAA + zero). Feb 3 / Feb 6 are computed too, purely as a
   drift-QC cross-check — *not applied*.

2. **C2H6 — cross-instrument peak-alignment (Section D). ⚠️ FLAGGED, LOW CONFIDENCE.**
   The tank has only one certified C2H6 point (NOAA, 1.63 ppb) — far below real plume
   levels — so no trustworthy tank-anchored fit is possible. Instead Ultra460 is treated
   as the C2H6 reference and Pico017/Ultra321 are matched to it by plume-peak magnitude.
   This rests on Ultra460's own C2H6 being correct, which is *assumed*, not verified.

### Reusable for a future campaign or a new species

Two generic entry points in `src/calibration.py` back both methods above — pick "tank"
or "reference instrument", pass your data, get coefficients back with the standard
checks already run: `cal.calibrate_and_check_tank(...)` and
`cal.calibrate_and_check_reference(..., anchor='ols'|'pinned')`. This notebook dogfoods
both (the CH4 Picarro cross-cal and the C2H6 fit below use `calibrate_and_check_reference`
directly); mobile-slv's own CH4/C3H8 tank fitting drops to the lower-level
`window_stats`/`fit_species` instead, since it needs the shared per-instrument window
stats for error bars and drift QC — see the module docstring for when to use which.

### How to read this notebook

Sections run **A → H in order**; each builds on the last. Cells labeled **sanity check**
prove a step did what it claims (points collapsing onto a 1:1 line, corrected traces
tracking each other, residuals flat). The **Ultra321 C2H6 problem** (poor fit, strong
C3H8 cross-interference) is surfaced with real diagnostic plots in Section D and carried
through to the "Open items" recap in Section H — it is **not** resolved here.

**Output** — `04_calibrated/` becomes the single complete Stage 04 directory:
- **Calibrated** (`Raw`+`Eng` for the four analyzers): gains `*_cal` columns and a
  `cal_coefs_ref` column pointing at `calibration_coefs.json`.
- **Passthrough** (Spectra/Spectralite + GPS/Anem/Sprinter/LGR): copied unchanged, no
  `cal_coefs_ref` column — that absence is how you tell passthrough from calibrated.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import importlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.insert(0, str(Path().resolve().parent))
from paths import (STAGE_02_DIR, STAGE_03_DIR, STAGE_04_DIR,
                   TANK_DETAILS_PATH, REPO_ROOT)
from src import calibration as cal
from src.provenance import check_clean, upstream_ref
importlib.reload(cal)   # pick up edits to src/calibration.py without restarting the kernel
from src.align import load_aligned_series

print('Imports OK — calibration module reloaded')

In [ ]:
# ── Campaign-specific configuration ───────────────────────────────────────────
# Where each instrument's aligned gas data lives in Stage 03, and the color it wears
# in every plot below. Colors are a presentation choice for THIS instrument set, so
# they live here (the generic plotting functions take `colors` as a parameter).
INSTRUMENTS = {
    'Picarro':  {'dir': 'WYO_picarro',        'subdir': ''},
    'Ultra460': {'dir': 'WYO_aerisultra460',  'subdir': 'Raw'},
    'Ultra321': {'dir': 'LANL_aerisultra321', 'subdir': 'Raw'},
    'Pico017':  {'dir': 'LANL_aerispico017',  'subdir': 'Raw'},
}
INST_COLORS = {
    'Picarro':  '#1f77b4',   # blue
    'Ultra460': '#ff7f0e',   # orange
    'Ultra321': '#2ca02c',   # green
    'Pico017':  '#d62728',   # red
}
CAL_DATE_CANONICAL = '20260212'   # only event spanning the full 0-57 ppm dilution ladder

def load_full_series(inst, col):
    """Concatenated Series for one instrument/column from all good Stage 03 aligned files."""
    cfg = INSTRUMENTS[inst]
    return load_aligned_series(STAGE_03_DIR, cfg['dir'], cfg['subdir'], col)

print('Instruments:', list(INSTRUMENTS))
print('Canonical calibration date:', CAL_DATE_CANONICAL)

---
## A — Parse the tank calibration manifest

`tank_details.txt` lists the certified concentration of each tank/dilution and the UTC
time windows during which each was delivered on each calibration date. `parse_tank_details`
returns `(TANK, WINDOWS_BY_DATE)`.

In [ ]:
TANK, WINDOWS_BY_DATE = cal.parse_tank_details(TANK_DETAILS_PATH)

print('Tank standards:', list(TANK), '\n')
for date, wins in sorted(WINDOWS_BY_DATE.items()):
    tag = '   <- canonical (full dilution ladder)' if date == CAL_DATE_CANONICAL else ''
    print(f'{date}: {len(wins)} windows{tag}')
    print('   ', [w['tank_key'] for w in wins])

The certified concentrations, as a table. Note the asymmetry that shapes the rest of
the notebook: **CH4 and C3H8 span a full ladder** (zero → 57 ppm / 10 ppm across five
dilutions + NOAA), while **C2H6 has exactly one certified value** (NOAA, 1.63 ppb) plus
the implicit zero. That single low point is why C2H6 cannot use the tank method.

In [ ]:
tank_df = pd.DataFrame(TANK).T[['CH4_ppm', 'C3H8_ppm', 'C2H6_ppb']]
tank_df.index.name = 'tank_key'
tank_df

---
## B — Load Stage 03 aligned data

One concatenated Series per instrument per species, pulled straight from the good
(non-`bad`, non-`bad_timestamp`) Stage 03 output. Ultra321 reports C2H6 in ppm; it is
converted to ppb here so all three C2H6 series share units.

In [ ]:
CH4  = {inst: load_full_series(inst, 'CH4_ppm') for inst in INSTRUMENTS}
C3H8 = {'Ultra321': load_full_series('Ultra321', 'C3H8_ppm')}          # only Ultra321 has C3H8
C2H6 = {
    'Ultra460': load_full_series('Ultra460', 'C2H6_ppb'),
    'Pico017':  load_full_series('Pico017', 'C2H6_ppb'),
    'Ultra321': load_full_series('Ultra321', 'C2H6_ppm') * 1000.0,     # ppm -> ppb
}

for inst, s in CH4.items():
    print(f'{inst:10s} CH4   {len(s):>9,} pts   {s.index[0]} -> {s.index[-1]}')
print()
for inst, s in C2H6.items():
    print(f'{inst:10s} C2H6  {len(s):>9,} pts')

**The "before" picture — a specific window at native resolution.** Rather than averaging
the whole campaign down to 10-minute means (which smears out exactly the plume structure
we care about), pick one time window and look at the raw 1 Hz data as stored. The default
centers on the largest CH4 excursion during an hour when all four instruments were
running; set `ZOOM_START` / `ZOOM_END` to any period you want to inspect. Section F
re-plots this *same window after* calibration, so the effect is directly comparable.
Nothing here is resampled, averaged, or interpolated.

In [ ]:
# ── Set your inspection window here ─────────────────────────────────────────────
# Leave both as None to auto-pick (largest ambient CH4 plume on a day when all four
# analyzers were co-deployed on WYO, tank windows excluded). Or set explicit UTC
# timestamps to inspect any period you want — this window is reused everywhere in the
# notebook that shows a native-resolution timeseries (here, the C2H6 checks in Section
# D/E, and the "did it help" check in Section F), so picking one custom range lets you
# watch the same period get calibrated end to end.
ZOOM_START = None   # e.g. pd.Timestamp('2026-02-11 20:30', tz='UTC')
ZOOM_END   = None   # e.g. ZOOM_START + pd.Timedelta('40min')

if ZOOM_START is None or ZOOM_END is None:
    with open(STAGE_02_DIR / 'routing_manifest.json') as _f:
        _route = json.load(_f)
    def _wyo_dates(pfx):
        return {'20' + k.split('_')[1] for k, v in _route.items() if v == 'WYO' and k.startswith(pfx)}
    _codeploy = _wyo_dates('Ultra100321') & _wyo_dates('Pico100017')   # dates all four were co-located
    _pic = CH4['Picarro'].dropna()
    _pic = _pic[_pic.index.strftime('%Y%m%d').isin(_codeploy)]
    for _wins in WINDOWS_BY_DATE.values():      # drop tank windows so the default is an ambient plume
        for _w in _wins:
            _t0 = pd.Timestamp(_w['start'], tz='UTC') - pd.Timedelta('5min')
            _t1 = pd.Timestamp(_w['end'],   tz='UTC') + pd.Timedelta('5min')
            _pic = _pic[(_pic.index < _t0) | (_pic.index > _t1)]
    ZOOM_CENTER = _pic.idxmax()
    ZOOM_START  = ZOOM_CENTER - pd.Timedelta('20min')
    ZOOM_END    = ZOOM_CENTER + pd.Timedelta('20min')

print('Inspection window:', ZOOM_START, '->', ZOOM_END)

fig = cal.plot_timeseries_panels(
    panels=[('CH4 — raw, all instruments', 'CH4 (ppm)', CH4),
            ('C2H6 — raw', 'C2H6 (ppb)', C2H6),
            ('C3H8 — Ultra321 raw', 'C3H8 (ppm)', C3H8)],
    colors=INST_COLORS, t0=ZOOM_START, t1=ZOOM_END,
    title=f'Raw Stage 03 data at native resolution — {ZOOM_START:%Y-%m-%d %H:%M}–{ZOOM_END:%H:%M} UTC')
fig.show()

---
## C — CH4 & C3H8 tank calibration (multi-point, tank-anchored)

A single multi-point OLS fit per instrument per date — **no** piecewise low/high split.
A single fit already reaches R² > 0.9997 for every instrument on Feb 12, and the small
residual wiggle that once motivated a piecewise split appears *identically* in Picarro
(the reference-grade, presumptively-linear instrument), which means it reflects tiny
imprecision in the dilution manifold's delivered concentrations, not instrument
nonlinearity. Splitting would just overfit that artifact.

First, **see what is being averaged**: the Feb 12 tank sequence with each delivery
window shaded and labeled. The flat plateaus inside the shaded spans are what
`window_stats` bins into a single mean per tank.

In [ ]:
feb12_windows = WINDOWS_BY_DATE[CAL_DATE_CANONICAL]
fig = cal.plot_timeseries_with_windows(
    CH4, feb12_windows, INST_COLORS,
    title='Feb 12 CH4 during the tank sequence — shaded spans are the fit windows',
    y_title='CH4 (ppm)')
fig.show()

In [ ]:
fig = cal.plot_timeseries_with_windows(
    {'Ultra321': C3H8['Ultra321']}, feb12_windows, INST_COLORS,
    title='Feb 12 C3H8 (Ultra321) during the tank sequence',
    y_title='C3H8 (ppm)')
fig.show()

Now the fits. `window_stats` computes each instrument's mean over each window;
`fit_species` regresses those means against the known tank concentration. Computed for
all three dates (Feb 12 canonical, Feb 3 / Feb 6 for drift QC).

In [ ]:
CH4_STATS  = {d: cal.window_stats(CH4, w)  for d, w in WINDOWS_BY_DATE.items()}
C3H8_STATS = {d: cal.window_stats(C3H8, w) for d, w in WINDOWS_BY_DATE.items()}

CH4_COEFS_BY_DATE  = {d: {inst: cal.fit_species(CH4_STATS[d], TANK, inst, 'CH4_ppm')
                          for inst in INSTRUMENTS} for d in WINDOWS_BY_DATE}
C3H8_COEFS_BY_DATE = {d: {'Ultra321': cal.fit_species(C3H8_STATS[d], TANK, 'Ultra321', 'C3H8_ppm')}
                      for d in WINDOWS_BY_DATE}

CH4_COEFS  = CH4_COEFS_BY_DATE[CAL_DATE_CANONICAL]
C3H8_COEFS = C3H8_COEFS_BY_DATE[CAL_DATE_CANONICAL]
print('Canonical (Feb 12) fits computed.')

In [ ]:
rows = []
for inst, c in CH4_COEFS.items():
    if c: rows.append({'gas': 'CH4', 'instrument': inst, **c})
for inst, c in C3H8_COEFS.items():
    if c: rows.append({'gas': 'C3H8', 'instrument': inst, **c})
pd.DataFrame(rows).set_index(['gas', 'instrument']).round(5)

### A second CH4 method, computed now so every check below can show both

In addition to the tank fit above, this notebook also cross-calibrates each Aeris unit's
CH4 directly against **Picarro's own continuous field CH4** — hundreds of thousands of
paired ambient points (nearest-timestamp match) instead of 7 discrete tank-window means.
Picarro itself has no cross-cal counterpart (it *is* the reference candidate) and stays
tank-only. Same idea C2H6 already uses (pick a well-behaved reference instrument,
cross-calibrate the others against it), with Picarro standing in for Ultra460 here.

Computed once, up front, so the sanity check, timeseries checks, and Section F can all
show tank-fit and cross-cal side by side. **Still comparison only — `CH4_COEFS` (the tank
fit) is what Sections E/F actually save and apply**, per your call to keep this separate
for now.

In [ ]:
PICARRO_DATES = sorted({d.strftime('%Y%m%d') for d in pd.DatetimeIndex(CH4['Picarro'].dropna().index).normalize().unique()})
CROSS_CAL_DATES = {
    'Ultra460': PICARRO_DATES,
    'Ultra321': sorted(_wyo_dates('Ultra100321')),
    'Pico017':  sorted(_wyo_dates('Pico100017')),
}
CH4_PICARRO_AMB = cal.restrict_series(CH4['Picarro'], PICARRO_DATES, WINDOWS_BY_DATE)

# Fit only here (no plot yet — that's in the "closer look" section below), via the same
# generic reference-cal entry point a future campaign would use for ANY reference-
# instrument calibration: cal.fit_reference_cal(..., anchor='ols').
CH4_CROSS_COEFS, CH4_CROSS_PAIRS = {}, {}
for inst, dates in CROSS_CAL_DATES.items():
    amb = cal.restrict_series(CH4[inst], dates, WINDOWS_BY_DATE)
    paired = cal.pair_series_nearest(CH4_PICARRO_AMB, amb, tolerance_s=1)
    CH4_CROSS_COEFS[inst] = cal.fit_reference_cal(paired['ref'].values, paired['target'].values, anchor='ols')
    CH4_CROSS_PAIRS[inst] = paired
    c = CH4_CROSS_COEFS[inst]
    print(f'{inst}: cross-cal vs Picarro  n={c["n"]:,}  slope={c["slope"]:.4f}  intercept={c["intercept"]:+.4f}  R2={c["r2"]:.5f}')

**The 1:1 view.** Points are window means; **error bars are ±1σ of the in-window noise**
(how steadily each instrument held during that tank — usually smaller than the marker).
The solid line is each instrument's fit; the dashed line is `y = x`. Distance from the
dashed line *is* the calibration error the fit corrects. A perfect instrument would already
sit on the dashed line.

In [ ]:
def scatter_xy(stats_df, tank, insts, species_key):
    """Build {inst: x/y/err arrays} for plot_calibration_scatter (err = 1σ in-window noise)."""
    xt = stats_df['tank_key'].map(lambda k: tank.get(k, {}).get(species_key)).astype(float)
    x, y, err = {}, {}, {}
    for inst in insts:
        if f'{inst}_mean' in stats_df.columns:
            x[inst] = xt.values
            y[inst] = stats_df[f'{inst}_mean'].astype(float).values
            if f'{inst}_std' in stats_df.columns:
                err[inst] = stats_df[f'{inst}_std'].astype(float).values
    return x, y, err

xg, yg, eg = scatter_xy(CH4_STATS[CAL_DATE_CANONICAL], TANK, INSTRUMENTS, 'CH4_ppm')
fig = cal.plot_calibration_scatter(xg, yg, CH4_COEFS, INST_COLORS,
        x_title='Tank CH4 (ppm)', y_title='Instrument CH4 (ppm)',
        title='CH4 calibration (Feb 12) — points ±1σ in-window noise, per-instrument fit, 1:1 line',
        yerr_by_group=eg)
fig.show()

In [ ]:
xg, yg, eg = scatter_xy(C3H8_STATS[CAL_DATE_CANONICAL], TANK, ['Ultra321'], 'C3H8_ppm')
fig = cal.plot_calibration_scatter(xg, yg, C3H8_COEFS, INST_COLORS,
        x_title='Tank C3H8 (ppm)', y_title='Ultra321 C3H8 (ppm)',
        title='C3H8 calibration (Feb 12) — Ultra321, points ±1σ in-window noise',
        yerr_by_group=eg)
fig.show()

**Sanity check.** Apply each fit back to its own window means. If the fit is good,
the corrected points should collapse onto the dashed 1:1 line and residuals should shrink
to near zero. The table reports the worst-case and RMS residual per instrument (in ppm).

In [ ]:
stats_df = CH4_STATS[CAL_DATE_CANONICAL]
xt = stats_df['tank_key'].map(lambda k: TANK.get(k, {}).get('CH4_ppm')).astype(float)
xg, yg, eg, resid_rows = {}, {}, {}, []
for inst, c in CH4_COEFS.items():
    if c is None:
        continue
    corrected = cal.apply_linear(stats_df[f'{inst}_mean'].astype(float), c)
    xg[inst], yg[inst] = xt.values, corrected.values
    if f'{inst}_std' in stats_df.columns:
        eg[inst] = (stats_df[f'{inst}_std'].astype(float) / c['slope']).values   # in-window noise carried through the fit
    r = (corrected - xt)
    resid_rows.append({'instrument': inst,
                       'max_abs_resid_ppm': float(np.nanmax(np.abs(r))),
                       'rms_resid_ppm': float(np.sqrt(np.nanmean(r ** 2)))})
fig = cal.plot_calibration_scatter(xg, yg, {}, INST_COLORS,
        x_title='Tank CH4 (ppm)', y_title='CORRECTED instrument CH4 (ppm)',
        title='CH4 sanity check — TANK FIT (applied to output)',
        yerr_by_group=eg)
fig.show()
print('Tank-fit residuals at the tank points:')
print(pd.DataFrame(resid_rows).set_index('instrument').round(4))

# Does the Picarro cross-cal ALSO get the tank points right? It was never trained on tank
# data, so this is a genuine out-of-sample check — with a real caveat: cross-cal training
# data is >99.9% ambient (<5 ppm); only ~0.04% of paired points exceed 10 ppm, essentially
# all from a single large plume event. So its high-concentration behavior rests on a
# handful of leverage points with real timestamp-alignment noise (matching two different
# instruments' response to a brief, sharp plume within a 1s tolerance), unlike the tank's
# sustained, controlled delivery — expect WORSE agreement than the tank fit at Dilution5
# (57 ppm), even though it's much tighter at ambient levels (see the comparison below).
xg2, yg2, cross_resid_rows = {}, {}, []
for inst, c in CH4_CROSS_COEFS.items():
    corrected = cal.apply_linear(stats_df[f'{inst}_mean'].astype(float), c)
    xg2[inst], yg2[inst] = xt.values, corrected.values
    r = (corrected - xt)
    cross_resid_rows.append({'instrument': inst,
                             'max_abs_resid_ppm': float(np.nanmax(np.abs(r))),
                             'rms_resid_ppm': float(np.sqrt(np.nanmean(r ** 2)))})
fig2 = cal.plot_calibration_scatter(xg2, yg2, {}, INST_COLORS,
        x_title='Tank CH4 (ppm)', y_title='CORRECTED instrument CH4 (ppm)',
        title='CH4 sanity check — PICARRO CROSS-CAL, comparison only (out-of-sample; expect worse at Dilution5)')
fig2.show()
print('Picarro-cross-cal residuals at the SAME tank points (out-of-sample check):')
print(pd.DataFrame(cross_resid_rows).set_index('instrument').round(4))

**Timeseries version of the same check, both methods.** The 1:1/sanity-check views above
are window means; this is both fits applied at **native resolution** over your inspection
window (`ZOOM_START`/`ZOOM_END`, set in Section B) — raw, tank fit (applied to output),
and Picarro cross-cal (comparison only). This is the "beginning" of calibration working;
Section F repeats this exact pattern as the "end," once the tank-fit correction has been
applied to the full on-disk output.

In [ ]:
cal_ch4_begin = {inst: cal.apply_linear(CH4[inst], c) for inst, c in CH4_COEFS.items() if c}
cal_ch4_begin_cross = {inst: cal.apply_linear(CH4[inst], c) for inst, c in CH4_CROSS_COEFS.items()}
cal_ch4_begin_cross['Picarro'] = CH4['Picarro']   # reference — already trusted/tank-calibrated

fig = cal.plot_timeseries_panels(
    panels=[('CH4 raw', 'CH4 (ppm)', CH4),
            ('CH4 calibrated — TANK FIT (applied to output)', 'CH4_cal (ppm)', cal_ch4_begin),
            ('CH4 calibrated — PICARRO CROSS-CAL (comparison only)', 'CH4_cal (ppm)', cal_ch4_begin_cross)],
    colors=INST_COLORS, t0=ZOOM_START, t1=ZOOM_END,
    title=f'CH4 — beginning-of-calibration check, both methods — {ZOOM_START:%Y-%m-%d %H:%M}–{ZOOM_END:%H:%M} UTC')
fig.show()

def _ch4_spread(series_dict, t0, t1):
    base_mask = lambda s: (s.index < t0 + pd.Timedelta('3min')) | (s.index > t1 - pd.Timedelta('3min'))
    base, peak = {}, {}
    for i, s in series_dict.items():
        sub = s[t0:t1]
        base[i] = float(sub[base_mask(sub)].median())
        peak[i] = float(sub.max())
    return max(base.values()) - min(base.values()), max(peak.values()) - min(peak.values())

tank_base, tank_peak = _ch4_spread(cal_ch4_begin, ZOOM_START, ZOOM_END)
cross_base, cross_peak = _ch4_spread(cal_ch4_begin_cross, ZOOM_START, ZOOM_END)
print(f'Tank fit:      baseline spread={tank_base:.3f} ppm   peak spread={tank_peak:.3f} ppm')
print(f'Picarro cross: baseline spread={cross_base:.3f} ppm   peak spread={cross_peak:.3f} ppm')

### A closer look — the Picarro cross-cal's own fit

`CH4_CROSS_COEFS` (computed earlier, right after the tank coefficient table, so every
check above could show both methods) regresses each Aeris unit's ambient CH4 directly
against **Picarro's own continuous field CH4** — hundreds of thousands of paired points
spanning the real ambient range, instead of 7 discrete tank-window means from one
afternoon. This is the "reference instrument" half of the module's two generic entry
points (`cal.fit_reference_cal` for the fit alone, `cal.calibrate_and_check_reference`
for fit + standard checks together) — the same functions a future campaign would reach
for.

Below: the residual-RMS comparison table, and one ambient-space scatter per instrument
(fit line + 1:1, via `calibrate_and_check_reference`) — contrast with the sanity-check
cells above, which evaluate both methods at the tank's 7 known points instead.

**Still comparison only — nothing here is applied to the saved output.** `CH4_COEFS`
(the tank fit) remains what Sections E/F actually calibrate with.

In [ ]:
# CH4_CROSS_COEFS / CH4_CROSS_PAIRS were already computed earlier (right after the tank
# coefficient table) so every check above could show both methods. This assembles the
# residual-RMS comparison — tank-fit RMS AT the tank points (from the sanity check above)
# vs cross-cal RMS over its own ambient training population (different populations; see
# the sanity-check cell above for both methods evaluated at the SAME tank points).
TANK_FIT_RMS = {r['instrument']: r['rms_resid_ppm'] for r in resid_rows}

comparison_rows = []
for inst, paired in CH4_CROSS_PAIRS.items():
    c = CH4_CROSS_COEFS[inst]
    cross_resid = cal.apply_linear(paired['target'], c) - paired['ref']
    comparison_rows.append({
        'instrument': inst,
        'tank_fit_RMS_ppm': TANK_FIT_RMS.get(inst, float('nan')),
        'cross_cal_RMS_ppm': float(np.sqrt(np.mean(cross_resid ** 2))),
        'cross_cal_R2': c['r2'],
        'cross_cal_n': c['n'],
    })

comparison_df = pd.DataFrame(comparison_rows).set_index('instrument').round(4)
print('Tank fit (@ 7 tank points) vs Picarro cross-cal (@ its ambient training population):')
comparison_df

In [ ]:
# Same generic entry point, now with its standard check (fit + scatter) via
# cal.calibrate_and_check_reference -- what a future campaign would call directly for a
# reference-instrument calibration. Refits internally (deterministically identical to
# CH4_CROSS_COEFS above); plotted on a 3,000-point subsample per instrument so the
# figures stay light -- the fit itself always uses the full paired set.
for inst, paired in CH4_CROSS_PAIRS.items():
    cal.calibrate_and_check_reference(
        paired['ref'].values, paired['target'].values, anchor='ols',
        colors=INST_COLORS, x_label='Picarro CH4 (ppm, ambient)',
        y_label='Instrument CH4 (ppm, ambient)', target_label=inst,
        title_prefix='CH4 cross-cal vs Picarro (comparison only) — ',
        plot_sample_size=3000, plot_seed=0)

### Drift QC — does one calibration hold for the whole campaign?

**What "drift" means:** an instrument's response can change over time (aging optics, temperature
history, etc.), so a calibration measured on one day slowly becomes *wrong* on later days. We
calibrate everything from the **Feb 12** tank event and apply those coefficients campaign-wide.
That is only justified if the instruments *didn't* drift between the tank events. If they did,
the single static calibration would inject a growing error the further you get from Feb 12.

We have three tank events (Feb 3, 6, 12) to test this with. Two complementary views:

1. **Apply the one Feb-12 calibration to *every* tank date.** If an instrument didn't drift,
   its corrected readings land on the 1:1 line on Feb 3 and Feb 6 too — not just on Feb 12
   (where it's guaranteed to, since that's the data it was fit on). Points pulling *off* the
   line on the earlier dates would be the visual signature of drift.

2. **Residual error vs the fit-noise floor.** The RMS error left after applying the Feb-12
   calibration to each date. The Feb-12 point is just the within-day fit noise (manifold
   delivery imprecision — the same wiggle that shows up even in reference-grade Picarro). If
   Feb 3 / Feb 6 sit no higher than that floor, there is no drift signal above the noise.

*(A complementary continuous check — tracking calibrated Picarro−Ultra460 agreement across all
WYO days — would live in the analysis repo; here we stay with the tank-anchored evidence. And
note all three tank events fall inside the WYO window Feb 3–12, so the Jan/March MML dates
remain an extrapolation regardless of what this QC shows.)*

*(The Picarro cross-cal computed above doesn't have a parallel "drift" view here — it's a
single fit pooled across all WYO-day overlap, not a series of discrete calibration events
to compare against each other the way the tank fit's three dates are. Its own robustness
concern is different in kind: very few high-concentration leverage points, not temporal
drift — see the sanity-check and comparison cells above.)*

In [ ]:
# Check 1 — apply the single canonical (Feb-12) calibration to every date's tank windows.
# On the 1:1 line across all three dates => the instrument did not drift. Error bars = 1σ
# in-window noise carried through the fit.
_date_color = {'20260203': '#9ecae1', '20260206': '#4292c6', '20260212': '#08519c'}
fig = make_subplots(rows=2, cols=2, subplot_titles=list(INSTRUMENTS))
_seen = set()
for idx, inst in enumerate(INSTRUMENTS):
    rr, cc = idx // 2 + 1, idx % 2 + 1
    c = CH4_COEFS[inst]
    for d in sorted(WINDOWS_BY_DATE):
        sd = CH4_STATS[d]
        col = f'{inst}_mean'
        if c is None or col not in sd.columns:
            continue
        xt = sd['tank_key'].map(lambda k: TANK.get(k, {}).get('CH4_ppm')).astype(float)
        yc = cal.apply_linear(sd[col].astype(float), c)          # Feb-12 coefficients
        scol = f'{inst}_std'
        ye = (sd[scol].astype(float) / c['slope']).values if scol in sd.columns else None
        fig.add_trace(go.Scatter(x=xt, y=yc, mode='markers',
                                 marker=dict(color=_date_color.get(d, 'gray'), size=8),
                                 error_y=(dict(type='data', array=ye, visible=True, thickness=1, width=2)
                                          if ye is not None else None),
                                 name=d, legendgroup=d, showlegend=d not in _seen), row=rr, col=cc)
        _seen.add(d)
    fig.add_trace(go.Scatter(x=[0, 60], y=[0, 60], mode='lines',
                             line=dict(color='rgba(0,0,0,0.4)', dash='dash'),
                             showlegend=False), row=rr, col=cc)
fig.update_layout(
    title='CH4: single Feb-12 calibration applied to every tank date — on 1:1 across dates = no drift',
    template='plotly_white', height=640)
fig.update_xaxes(title_text='true tank CH4 (ppm)')
fig.update_yaxes(title_text='corrected (ppm)')
fig.show()

In [ ]:
# Check 2 — RMS error (ppm) from using the single Feb-12 calibration on each date.
# The Feb-12 marker is just the within-day fit-noise floor (calibration applied to its own
# data); if Feb 3 / Feb 6 sit no higher, there's no drift signal above the noise.
dates = sorted(WINDOWS_BY_DATE)
drift_rows = []
fig = go.Figure()
for inst in INSTRUMENTS:
    c = CH4_COEFS[inst]
    if c is None:
        continue
    ys = []
    for d in dates:
        sd = CH4_STATS[d]
        col = f'{inst}_mean'
        if col not in sd.columns:
            ys.append(np.nan); continue
        xt = sd['tank_key'].map(lambda k: TANK.get(k, {}).get('CH4_ppm')).astype(float)
        err = (cal.apply_linear(sd[col].astype(float), c) - xt).values
        err = err[np.isfinite(err)]                      # drop dates the instrument didn't run (e.g. Pico Feb 3)
        rms = float(np.sqrt(np.mean(err ** 2))) if err.size else np.nan
        ys.append(rms)
        drift_rows.append({'instrument': inst, 'date': d, 'rms_err_ppm': rms})
    fig.add_trace(go.Scatter(x=dates, y=ys, mode='lines+markers', name=inst,
                             line=dict(color=INST_COLORS[inst])))
fig.update_layout(
    title='CH4 drift: RMS error from the single Feb-12 calibration, per tank date (flat & low = no drift)',
    xaxis_title='Tank date', yaxis_title='RMS(corrected − true)  [ppm]',
    template='plotly_white', height=420)
fig.add_annotation(text='Feb-12 = fit-noise floor', showarrow=False,
                   xref='paper', yref='paper', x=0.99, y=0.99, xanchor='right', font=dict(color='gray'))
fig.show()
pd.DataFrame(drift_rows).pivot(index='instrument', columns='date', values='rms_err_ppm').round(3)

---
## D — C2H6 cross-instrument peak-alignment  ⚠️ FLAGGED — lower confidence

Not tank-anchored. Ultra460 is treated as the C2H6 reference; Pico017 and Ultra321 are
calibrated by matching plume-peak magnitudes against it, over every WYO co-deployment day
(derived from `routing_manifest.json`, not hardcoded), with the tank windows excluded.
Every coefficient here is tagged `confidence: "low"` in the saved output.

First, **why the tank method is unusable for C2H6**: the ambient plume distribution sits
far above the single certified point.

In [ ]:
# Why C2H6 cannot be tank-calibrated: there is exactly ONE certified nonzero tank point
# (NOAA, 1.63 ppb), plus the implicit zero. But the signal we actually need to calibrate
# is plume peaks of tens to thousands of ppb. Below, the one tank point vs the levels that
# matter — on a log axis. Anchoring a slope on a single point sitting a decade below even
# the typical ambient baseline (and three decades below big plumes) is not defensible, so
# C2H6 is cross-calibrated against Ultra460 instead (the method in the rest of Section D).
u460 = C2H6['Ultra460'].dropna()
levels = [
    ('NOAA tank — only certified point', TANK['NOAA']['C2H6_ppb'], 'black'),
    ('ambient median',                    float(u460.median()),      '#8a8a8a'),
    ('ambient 99th percentile',           float(u460.quantile(0.99)),'#8a8a8a'),
    ('plume-peak fit threshold (50 ppb)', 50.0,                      'gray'),
    ('largest plume peak',                float(u460.max()),         INST_COLORS['Ultra460']),
]
fig = go.Figure()
for name, val, color in levels:
    fig.add_trace(go.Scatter(x=[val], y=[name], mode='markers+text',
                             marker=dict(color=color, size=13),
                             text=[f'  {val:,.2f} ppb'], textposition='middle right', showlegend=False))
fig.update_xaxes(type='log', title_text='C2H6 (ppb, log scale)')
fig.update_layout(title='Why C2H6 cannot be tank-calibrated: one tank point vs the range we must calibrate',
                  template='plotly_white', height=340, margin=dict(l=230))
fig.show()

In [ ]:
with open(STAGE_02_DIR / 'routing_manifest.json') as f:
    ROUTING = json.load(f)
WYO_DATES = sorted({f'20{k.split("_")[1]}' for k, v in ROUTING.items() if v == 'WYO'})
print(f'WYO co-deployment dates ({len(WYO_DATES)}):', WYO_DATES)

CAL_WINDOW_PAD_MIN = 5   # pad the tank windows so tank gas can't leak into the ambient fit
CAL_EXCLUDE_RANGES = [
    (pd.Timestamp(w['start'], tz='UTC') - pd.Timedelta(minutes=CAL_WINDOW_PAD_MIN),
     pd.Timestamp(w['end'], tz='UTC')   + pd.Timedelta(minutes=CAL_WINDOW_PAD_MIN))
    for wins in WINDOWS_BY_DATE.values() for w in wins
]

def restrict_to_wyo_ambient(series):
    """WYO-day ambient only: drop non-WYO dates and the (padded) tank windows."""
    if series is None:
        return None
    s = series[series.index.strftime('%Y%m%d').isin(WYO_DATES)]
    mask = pd.Series(True, index=s.index)
    for t0, t1 in CAL_EXCLUDE_RANGES:
        mask &= ~((s.index >= t0) & (s.index <= t1))
    return s[mask]

U460_AMB      = restrict_to_wyo_ambient(C2H6['Ultra460'])
PICO_AMB      = restrict_to_wyo_ambient(C2H6['Pico017'])
U321_AMB      = restrict_to_wyo_ambient(C2H6['Ultra321'])
CH4_PIC_AMB   = restrict_to_wyo_ambient(CH4['Picarro'])
C3H8_U321_AMB = restrict_to_wyo_ambient(C3H8['Ultra321'])
print(f'Ultra460 ambient C2H6 points: {len(U460_AMB):,}')

`find_peak_matches` locates plume peaks in the Ultra460 reference (per day) and, at
each peak time, grabs the local max of every other series within a ±10 s window — Pico017
and Ultra321 as calibration targets, plus Picarro CH4 and Ultra321 C3H8 as interference
diagnostics.

In [ ]:
PEAK_HEIGHT_PPB     = 50.0
PEAK_PROMINENCE_PPB = 15.0
PEAK_MIN_DISTANCE_S = 30
PEAK_MATCH_WINDOW_S = 10

peaks_df = cal.find_peak_matches(
    U460_AMB,
    {'Pico017': PICO_AMB, 'Ultra321': U321_AMB,
     'CH4_picarro': CH4_PIC_AMB, 'C3H8_ultra321': C3H8_U321_AMB},
    height=PEAK_HEIGHT_PPB, prominence=PEAK_PROMINENCE_PPB,
    min_distance_s=PEAK_MIN_DISTANCE_S, window_s=PEAK_MATCH_WINDOW_S)
print(f'Plume peaks found: {len(peaks_df)}')
peaks_df.groupby('date').size().rename('n_peaks').to_frame()

**See what the peak finder did — and meet the Ultra321 problem.** One clean co-deployment
day (busiest ambient day that isn't a tank-cal day). Ultra460 (reference) and Pico017
share a scale and their plume peaks line up — good. **Ultra321 is plotted on its own axis
below**, because it can't share one with the others: its C2H6 baseline sits near
**−135 ppb** and it reads negative ~89% of the campaign. That broken baseline — on top of
the C3H8 cross-talk shown later — is the root of its failed fit.

In [ ]:
# Pick the busiest AMBIENT co-deployment day (exclude tank-cal dates) for a clean illustration
_cal_dates = set(WINDOWS_BY_DATE)
_by_day = peaks_df.groupby('date').size()
_ambient = _by_day[[d.replace('-', '') not in _cal_dates for d in _by_day.index]]
example_date = (_ambient if len(_ambient) else _by_day).idxmax()
day = pd.Timestamp(example_date).strftime('%Y-%m-%d')
peak_times = peaks_df.loc[peaks_df['date'] == example_date, 'peak_time']

# Ultra460 (reference) + Pico017 — comparable scale, peaks should line up.
fig = cal.plot_peaks_highlighted(
    {'Ultra460': U460_AMB[day], 'Pico017': PICO_AMB[day]}, peak_times, INST_COLORS,
    title=f'C2H6 plume peaks on {example_date} — Ultra460 (ref) & Pico017 (x = matched peaks)',
    y_title='C2H6 (ppb)')
fig.show()

# Ultra321 on its OWN axis — note the large negative baseline (why it can't share the plot).
fig = cal.plot_peaks_highlighted(
    {'Ultra321': U321_AMB[day]}, peak_times, INST_COLORS,
    title=f'Ultra321 C2H6 on {example_date} — SEPARATE AXIS: broken (negative) baseline near -135 ppb',
    y_title='C2H6 (ppb)')
fig.show()

**The zero+span fit.** Two anchors: the **baseline** is fixed first — matched directly to
Ultra460's own ambient level, not the tank — and the **plume peaks** fix the gain/span
against Ultra460 second. This is the "reference instrument" entry point
(`cal.calibrate_and_check_reference`, `anchor='pinned'`) — the same generic function the
CH4 Picarro cross-cal above uses with `anchor='ols'` instead. Below, one scatter per
target (target peak vs Ultra460 peak, fit line + 1:1) — Pico017 tracks tightly, Ultra321
scatters.

> **Why match Ultra460's baseline instead of the tank's absolute zero.** An earlier
> version of this fit anchored to the certified tank N2-zero instead. That's the more
> "physically true" anchor, but it left Pico017's corrected baseline sitting ~5–6 ppb
> *below* Ultra460's, because the tank says Pico017's true background C2H6 ≈ 0 while
> Ultra460 reads ~+5–7 ambient — the two instruments genuinely disagree at baseline. Since
> the entire point of this section is cross-instrument agreement with Ultra460 (not an
> absolute-truth measurement), **the baseline anchor is set to match Ultra460's own
> ambient level first**, by construction, and the gain is fit on top of that. This means
> the corrected reading is not independently traceable to the certified zero — a
> deliberate trade, made explicit here and in the saved coefficients' `note` field.

In [ ]:
# Zero + span, via the generic reference-cal entry point (cal.calibrate_and_check_reference,
# anchor='pinned'): the baseline anchor (median of the WYO-ambient population, tank
# windows excluded -- the same U460_AMB/PICO_AMB/U321_AMB series used for peak-matching
# above) sets each target's zero to match Ultra460's own ambient level FIRST; the plume
# peaks set the gain/span SECOND. See the markdown above for why this anchor was chosen
# over the tank's absolute zero. ambient_baseline_stats also reports the ambient spread
# (half-IQR, robust to occasional plumes) as an error bar on the anchor.
Z_REF, Z_REF_SPREAD, _ = cal.ambient_baseline_stats(U460_AMB, q=0.5)
C2H6_BASELINE = {'Ultra460': (Z_REF, Z_REF_SPREAD)}
C2H6_COEFS = {}
for inst, amb in [('Pico017', PICO_AMB), ('Ultra321', U321_AMB)]:
    sub = peaks_df.dropna(subset=['ref', inst])
    z_tgt, z_tgt_spread, _ = cal.ambient_baseline_stats(amb, q=0.5)
    C2H6_BASELINE[inst] = (z_tgt, z_tgt_spread)
    c = cal.calibrate_and_check_reference(
        sub['ref'].values, sub[inst].values, anchor='pinned', z_ref=Z_REF, z_tgt=z_tgt,
        colors=INST_COLORS, x_label='Ultra460 C2H6 peak (ppb)',
        y_label='Instrument C2H6 peak (ppb)', target_label=inst,
        title_prefix='C2H6 zero+span vs Ultra460 (FLAGGED) — ')
    c['z_ref_std'], c['z_tgt_std'] = Z_REF_SPREAD, z_tgt_spread
    C2H6_COEFS[inst] = c
    print(f"{inst}:  ambient baseline {z_tgt:+7.1f}±{z_tgt_spread:.1f} -> {Z_REF:+.1f}±{Z_REF_SPREAD:.1f} ppb   "
          f"gain={c['gain']:.4f}  slope={c['slope']:.4f}  intercept={c['intercept']:+.2f}  "
          f"R2(span)={c['r2']:.4f}  n={c['n']}")

**Timeseries version, same custom window as everywhere else.** The scatter above is
peak-only; this shows the fit applied at native resolution over your `ZOOM_START`/
`ZOOM_END` window (set in Section B) — Ultra460 (reference) plotted against each target's
raw and corrected trace. Since the anchor is now the ambient baseline, the two traces
should sit on Ultra460's *baseline* here even outside a plume — that's the direct visual
proof the new anchor does what it's supposed to.

In [ ]:
pico_corr_begin = cal.apply_linear(C2H6['Pico017'], C2H6_COEFS['Pico017'])
u321_corr_begin = cal.apply_linear(C2H6['Ultra321'], C2H6_COEFS['Ultra321'])
_cmp_colors = {'Ultra460 (reference)': INST_COLORS['Ultra460'],
               'Pico017': INST_COLORS['Pico017'], 'Ultra321': INST_COLORS['Ultra321']}
fig = cal.plot_raw_corrected_vs_reference(
    C2H6['Ultra460'], 'Ultra460 (reference)',
    {'Pico017': (C2H6['Pico017'], pico_corr_begin), 'Ultra321': (C2H6['Ultra321'], u321_corr_begin)},
    _cmp_colors, ZOOM_START, ZOOM_END,
    title=f'C2H6 — beginning-of-calibration check — {ZOOM_START:%Y-%m-%d %H:%M}–{ZOOM_END:%H:%M} UTC',
    y_title='C2H6 (ppb)')
fig.show()

> ### ⚠️ KNOWN OPEN ISSUE — Ultra321 C2H6 (not resolved in this notebook)
>
> The ambient-baseline anchor fixes Ultra321's *baseline* to match Ultra460 by
> construction — so the earlier "reads negative 89% of the time" problem is masked at
> baseline. But its **peaks remain unreliable**: R²(span) ≈ 0.83 (worse than Pico017's
> ≈ 0.998), and the fit **residual correlates ≈ +0.95 with C3H8** (diagnostic below) — the
> signature of **C3H8 leaking into the C2H6 retrieval**. No anchor choice fixes a spectral
> interference. The honest recommendation is still to **drop** the Ultra321 C2H6
> correction; it is retained tagged `confidence: "low"` only so the decision stays
> explicit. Carried forward to "Open items" in Section H.

**The interference diagnostic.** If a fit residual trends with CH4 or C3H8 level, that
channel is contaminating the C2H6 retrieval. Pico017 should scatter flat around zero;
Ultra321 vs C3H8 is the smoking gun.

In [ ]:
for inst in ['Pico017', 'Ultra321']:
    c = C2H6_COEFS[inst]
    sub = peaks_df.dropna(subset=['ref', inst])
    resid = sub[inst].values - (c['slope'] * sub['ref'].values + c['intercept'])
    for diag_col in ['CH4_picarro', 'C3H8_ultra321']:
        fig, corr = cal.plot_residual_diagnostic(
            resid, sub[diag_col].values, f'{diag_col} at peak', color=INST_COLORS[inst])
        fig.update_layout(title=f'{inst}: ' + fig.layout.title.text)
        fig.show()

**Raw vs corrected on the example day**, Pico017 (good fit) next to Ultra321 (poor
fit), so the quality gap is visible in the actual signal, not just in a scatter R².
(Applied here in ppb-fit space for illustration — the saved Ultra321 correction carries
`scale_in = 1000` because it acts on the native `C2H6_ppm` column.)

In [ ]:
for inst, amb in [('Pico017', PICO_AMB), ('Ultra321', U321_AMB)]:
    raw = amb[day]
    corrected = cal.apply_linear(raw, C2H6_COEFS[inst])
    fig = cal.plot_raw_vs_corrected(raw, corrected,
            f'{inst} C2H6 raw vs corrected — {example_date}', 'C2H6 (ppb)')
    fig.show()

---
## E — Assemble and save `calibration_coefs.json`

Each correction records the formula inputs (`slope`, `intercept`, `scale_in`), the
`col_in` → `col_out` mapping, and a `method`/`confidence` tag. The apply formula is
`calibrated = (measured * scale_in - intercept) / slope`.

In [ ]:
check_clean(REPO_ROOT, context='Stage 04')
git_hash, git_dirty = cal.git_info(REPO_ROOT)

corrections = []
for inst, c in CH4_COEFS.items():
    if c is None:
        continue
    corrections.append({'gas': 'CH4', 'instrument': inst, 'col_in': 'CH4_ppm', 'col_out': 'CH4_ppm_cal',
                        'scale_in': 1.0, 'method': 'tank_multipoint', 'confidence': 'high',
                        'cal_date': CAL_DATE_CANONICAL, 'slope': c['slope'], 'intercept': c['intercept'],
                        'r2': c['r2'], 'n_points': c['n']})
for inst, c in C3H8_COEFS.items():
    if c is None:
        continue
    corrections.append({'gas': 'C3H8', 'instrument': inst, 'col_in': 'C3H8_ppm', 'col_out': 'C3H8_ppm_cal',
                        'scale_in': 1.0, 'method': 'tank_multipoint', 'confidence': 'high',
                        'cal_date': CAL_DATE_CANONICAL, 'slope': c['slope'], 'intercept': c['intercept'],
                        'r2': c['r2'], 'n_points': c['n']})
for inst, c in C2H6_COEFS.items():
    col_in   = 'C2H6_ppb' if inst == 'Pico017' else 'C2H6_ppm'
    scale_in = 1.0 if inst == 'Pico017' else 1000.0
    corrections.append({'gas': 'C2H6', 'instrument': inst, 'col_in': col_in, 'col_out': 'C2H6_ppb_cal',
                        'scale_in': scale_in, 'method': 'zero_span_vs_ultra460', 'confidence': 'low',
                        'note': ('Baseline anchored to Ultra460 own ambient median (NOT the certified tank '
                                 'zero) so the corrected reading matches Ultra460 by construction; span/gain '
                                 'fit to plume peaks vs Ultra460. Ultra460 is the reference throughout, not '
                                 'independently tank-validated over the plume range (only NOAA=1.63 ppb '
                                 'certified). Because the anchor is ambient-matched rather than tank-zeroed, '
                                 'this correction is NOT independently traceable to the certified zero.'),
                        'reference_instrument': 'Ultra460', 'n_peaks': c['n'],
                        'baseline_method': 'ambient_median',
                        'baseline_ref_ppb': c['z_ref'], 'baseline_ref_spread_ppb': c['z_ref_std'],
                        'baseline_target_ppb': c['z_tgt'], 'baseline_target_spread_ppb': c['z_tgt_std'],
                        'gain': c['gain'], 'slope': c['slope'], 'intercept': c['intercept'], 'r2': c['r2']})

coefs_out = {
    'metadata': {
        'generated_utc': datetime.now(timezone.utc).isoformat(),
        'git_hash': git_hash, 'git_dirty': git_dirty,
        'upstream': {
            'wyo': upstream_ref(STAGE_03_DIR / 'apply_manifest_wyo.json'),
            'mml': upstream_ref(STAGE_03_DIR / 'apply_manifest_mml.json'),
        },
        'formula': 'calibrated = (measured * scale_in - intercept) / slope',
        'cal_date_canonical': CAL_DATE_CANONICAL,
        'drift_qc_dates': sorted(WINDOWS_BY_DATE),
        'c2h6_method_note': ('C2H6 uses a zero+span cross-cal: ambient-baseline anchor (matches Ultra460 '
                             'ambient median, NOT the tank zero) + plume-peak gain vs Ultra460. Ultra460 '
                             'C2H6 itself is passed through uncalibrated (the reference).'),
    },
    'corrections': corrections,
}
STAGE_04_DIR.mkdir(parents=True, exist_ok=True)
coefs_path = STAGE_04_DIR / 'calibration_coefs.json'
with open(coefs_path, 'w') as f:
    json.dump(coefs_out, f, indent=2)
print(f'Saved {len(corrections)} corrections -> {coefs_path}')

In [ ]:
pd.DataFrame(corrections)[['gas', 'instrument', 'col_in', 'col_out',
                            'method', 'confidence', 'slope', 'intercept', 'r2']].round(4)

**Does the C2H6 correction land on the reference?** With the ambient-baseline anchor, both
the *baseline* and the *peaks* are matched to Ultra460. One panel each (Ultra460 solid,
raw dotted, corrected solid), over the busiest cluster of plumes on the clean
co-deployment day:

- **Pico017** — raw runs ~+15–24 ppb high; after correction the **baseline and the peaks
  both track Ultra460**. This is a clean, trustworthy correction.
- **Ultra321** — the correction removes its huge offset and its baseline now matches
  Ultra460 too, but the channel is still broken at the peaks (C3H8 interference, poor
  span R²), so it's not trustworthy on any window despite the baseline looking fine.

In [ ]:
# Compare the C2H6 corrections against the reference they were fit to (Ultra460):
# raw vs corrected for Pico017 and Ultra321. Window = the densest cluster of plume peaks on
# the clean co-deployment day (most activity — not a single peak cherry-picked for outcome).
_pt = peaks_df.loc[peaks_df['date'] == example_date, 'peak_time'].sort_values()
_center = max(_pt, key=lambda t: ((_pt >= t - pd.Timedelta('15min')) & (_pt <= t + pd.Timedelta('15min'))).sum())
C2H6_WIN_START = _center - pd.Timedelta('15min')
C2H6_WIN_END   = _center + pd.Timedelta('15min')

pico_corr = cal.apply_linear(C2H6['Pico017'], C2H6_COEFS['Pico017'])   # ppb-space fit (scale_in=1)
u321_corr = cal.apply_linear(C2H6['Ultra321'], C2H6_COEFS['Ultra321'])
_cmp_colors = {'Ultra460 (reference)': INST_COLORS['Ultra460'],
               'Pico017': INST_COLORS['Pico017'], 'Ultra321': INST_COLORS['Ultra321']}
fig = cal.plot_raw_corrected_vs_reference(
    C2H6['Ultra460'], 'Ultra460 (reference)',
    {'Pico017': (C2H6['Pico017'], pico_corr), 'Ultra321': (C2H6['Ultra321'], u321_corr)},
    _cmp_colors, C2H6_WIN_START, C2H6_WIN_END,
    title=f'C2H6 raw vs corrected against the Ultra460 reference — {example_date}',
    y_title='C2H6 (ppb)')
fig.show()

---
## F — Apply calibration → `04_calibrated/`

`apply_calibration_to_dir` reads every good Stage 03 file (`Raw`+`Eng`; `bad/` and
`bad_timestamp/` are not descended into), adds the relevant `*_cal` columns and a
`cal_coefs_ref` column, and writes calibrated Parquet mirroring the Stage 03 layout.
Safe to re-run.

In [ ]:
CORR_BY_INST = {}
for c in corrections:
    CORR_BY_INST.setdefault(c['instrument'], {})[c['gas']] = c

APPLY_SUBDIRS = {
    'Picarro':  ('WYO_picarro', ['']),
    'Ultra460': ('WYO_aerisultra460', ['Raw', 'Eng']),
    'Ultra321': ('LANL_aerisultra321', ['Raw', 'Eng']),
    'Pico017':  ('LANL_aerispico017', ['Raw', 'Eng']),
}
apply_stats = {}
for inst, (inst_dir, subdirs) in APPLY_SUBDIRS.items():
    n_files = n_rows = 0
    for subdir in subdirs:
        src = STAGE_03_DIR / inst_dir / subdir if subdir else STAGE_03_DIR / inst_dir
        dst = STAGE_04_DIR / inst_dir / subdir if subdir else STAGE_04_DIR / inst_dir
        nf, nr = cal.apply_calibration_to_dir(src, dst, CORR_BY_INST.get(inst, {}))
        n_files += nf; n_rows += nr
    apply_stats[inst] = {'files': n_files, 'rows': n_rows}
    print(f'{inst:10s} {n_files:>4} files  {n_rows:>10,} rows -> {STAGE_04_DIR / inst_dir}')

apply_manifest = {'stage': '04_apply_calibration', 'run_utc': datetime.now(timezone.utc).isoformat(),
                  'git_hash': git_hash, 'git_dirty': git_dirty, 'coefs_source': str(coefs_path),
                  'instruments': apply_stats}
with open(STAGE_04_DIR / 'apply_manifest.json', 'w') as f:
    json.dump(apply_manifest, f, indent=2)
print('\nApply complete.')

**Sanity check 1 — on-disk output.** Read one calibrated file back off disk and
overlay its raw `CH4_ppm` against the written `CH4_ppm_cal`. This verifies the *actual
files*, not just the in-memory math.

In [ ]:
sample = sorted((STAGE_04_DIR / 'WYO_aerisultra460' / 'Raw').glob('*.parquet'))
if sample:
    df = pd.read_parquet(sample[0])
    fig = cal.plot_raw_vs_corrected(df['CH4_ppm'], df['CH4_ppm_cal'],
            f'Ultra460 CH4 raw vs calibrated (read back from disk) — {sample[0].name}', 'CH4 (ppm)')
    fig.show()
else:
    print('No calibrated Ultra460 files found — run the apply cell above first.')

**Sanity check 2 — magnitude of every correction.** Campaign-wide mean/std of
`(corrected − raw)` per correction, in output units. Nothing here should be physically
absurd (CH4 shifts of a few tenths of a ppm, etc.). `raw / scale_in` converts the
in-memory series into `col_in` units before applying, so the C2H6/Ultra321 unit twist is
handled correctly.

In [ ]:
srcmap = {'CH4': CH4, 'C3H8': C3H8, 'C2H6': C2H6}
rows = []
for c in corrections:
    raw = srcmap[c['gas']].get(c['instrument'])
    if raw is None:
        continue
    corrected = cal.apply_linear(raw / c['scale_in'], c)   # raw/scale_in -> col_in units
    delta = corrected - raw
    rows.append({'gas': c['gas'], 'instrument': c['instrument'], 'col_out': c['col_out'],
                 'status': 'applied', 'mean_delta': float(delta.mean()), 'std_delta': float(delta.std())})

# Comparison only — what the Picarro cross-cal WOULD produce for CH4 (NOT applied to output)
for inst, c in CH4_CROSS_COEFS.items():
    corrected = cal.apply_linear(CH4[inst], c)
    delta = corrected - CH4[inst]
    rows.append({'gas': 'CH4', 'instrument': inst, 'col_out': 'CH4_ppm_cal (cross-cal)',
                 'status': 'comparison only — NOT applied',
                 'mean_delta': float(delta.mean()), 'std_delta': float(delta.std())})

pd.DataFrame(rows).round(3)

**Sanity check 3 — did calibration actually help?** The *same window* as Section B
(`ZOOM_START`/`ZOOM_END`), at native resolution: raw vs calibrated, for CH4 (both the
applied tank fit and the comparison-only Picarro cross-cal, all four instruments) and
C2H6 (the three C2H6 channels). This is the "end" bookend to the "beginning" checks in
Sections C and D — same window, same instruments, before vs after the full correction.
Raw traces sit offset from one another; after calibration they should collapse toward a
common value (CH4), or the corrected Pico017/Ultra321 traces should sit on the
uncalibrated Ultra460 reference (C2H6) — that convergence, on real plume structure rather
than a smeared average, is the whole point of calibration.

In [ ]:
raw_ch4 = CH4                                        # uncalibrated, as loaded
cal_ch4 = {inst: cal.apply_linear(CH4[inst], c)      # per-point linear correction, native res
           for inst, c in CH4_COEFS.items() if c}
cal_ch4_cross_end = {inst: cal.apply_linear(CH4[inst], c) for inst, c in CH4_CROSS_COEFS.items()}
cal_ch4_cross_end['Picarro'] = CH4['Picarro']        # reference — already trusted/tank-calibrated

raw_c2h6 = C2H6
cal_c2h6 = {inst: cal.apply_linear(C2H6[inst], c) for inst, c in C2H6_COEFS.items()}
cal_c2h6['Ultra460'] = C2H6['Ultra460']              # reference — passthrough, uncalibrated

fig = cal.plot_timeseries_panels(
    panels=[('CH4 raw — instruments offset', 'CH4 (ppm)', raw_ch4),
            ('CH4 calibrated — TANK FIT, applied — should converge', 'CH4_cal (ppm)', cal_ch4),
            ('CH4 calibrated — PICARRO CROSS-CAL, comparison only', 'CH4_cal (ppm)', cal_ch4_cross_end),
            ('C2H6 raw', 'C2H6 (ppb)', raw_c2h6),
            ('C2H6 calibrated — Pico017/Ultra321 should sit on Ultra460', 'C2H6_cal (ppb)', cal_c2h6)],
    colors=INST_COLORS, t0=ZOOM_START, t1=ZOOM_END,
    title=f'End-of-calibration check — {ZOOM_START:%Y-%m-%d %H:%M}–{ZOOM_END:%H:%M} UTC')
fig.show()

---
## G — Passthrough: everything else, unchanged

Straight copies (no recomputation) of Stage 03 good data that calibration doesn't touch:
Spectra/Spectralite for the three Aeris instruments (no concentration columns), and
GPS/Anem/Sprinter/LGR in full (no tank or cross-cal coverage — LGR wasn't deployed until
Mar 10, after all three cal events). These get **no** `cal_coefs_ref` column, which is how
to tell them apart from calibrated files.

In [ ]:
passthrough_stats = {}
SPECTRA_SUBDIR = {
    'Ultra460': ('WYO_aerisultra460', 'Spectralite'),
    'Ultra321': ('LANL_aerisultra321', 'Spectra'),
    'Pico017':  ('LANL_aerispico017', 'Spectra'),
}
for inst, (inst_dir, subdir) in SPECTRA_SUBDIR.items():
    n = cal.copy_passthrough_dir(STAGE_03_DIR / inst_dir / subdir, STAGE_04_DIR / inst_dir / subdir)
    passthrough_stats[f'{inst_dir}/{subdir}'] = n
    print(f'{inst_dir}/{subdir:12s} {n:>4} files')

for inst_dir in ['LANL_GPS', 'LANL_Anem', 'WYO_sprinter', 'UOU_LGR']:
    n = cal.copy_passthrough_dir(STAGE_03_DIR / inst_dir, STAGE_04_DIR / inst_dir)
    passthrough_stats[inst_dir] = n
    print(f'{inst_dir:24s} {n:>4} files')

with open(STAGE_04_DIR / 'passthrough_manifest.json', 'w') as f:
    json.dump({'stage': '04_passthrough', 'run_utc': datetime.now(timezone.utc).isoformat(),
               'note': ('Straight copies of Stage 03 good data with no calibration applied — '
                        'either no concentration columns (Spectra) or no tank/cross-cal coverage '
                        '(GPS, Anem, Sprinter, LGR). No cal_coefs_ref column added.'),
               'copied': passthrough_stats}, f, indent=2)
print('\nPassthrough complete.')

---
## H — Reconciliation & open items

**File-count reconciliation.** Every good Stage 03 file for the calibrated instruments
should appear in Stage 04. Any `MISMATCH` flag means something was silently dropped.

In [ ]:
def count_parquet(d):
    d = Path(d)
    return len(list(d.glob('*.parquet'))) if d.exists() else 0

print('Calibrated instruments (Stage 03 -> Stage 04 direct-child file counts):')
for inst, (inst_dir, subdirs) in APPLY_SUBDIRS.items():
    for subdir in subdirs:
        s3 = STAGE_03_DIR / inst_dir / subdir if subdir else STAGE_03_DIR / inst_dir
        s4 = STAGE_04_DIR / inst_dir / subdir if subdir else STAGE_04_DIR / inst_dir
        a, b = count_parquet(s3), count_parquet(s4)
        label = f'{inst_dir}/{subdir or "."}'
        flag = '' if a == b else '   <-- MISMATCH'
        print(f'  {label:40s} 03={a:>4}  04={b:>4}{flag}')

with open(STAGE_04_DIR / 'apply_manifest.json') as f:
    print('\napply_manifest instruments:', json.load(f)['instruments'])
with open(STAGE_04_DIR / 'passthrough_manifest.json') as f:
    print('passthrough_manifest copied:', json.load(f)['copied'])

### Open items

- **Ultra321 C2H6 (Section D) — recommend dropping.** The ambient-baseline anchor fixes
  its baseline to match Ultra460 by construction, but its peaks remain unreliable
  (R²(span) ≈ 0.83) and the fit residual correlates ≈ +0.95 with C3H8 (cross-interference)
  — no anchor choice fixes a spectral interference. Currently written to output as
  `confidence: "low"`; the leaning is to **drop** it.
- **C2H6 baseline is anchored to Ultra460, not to the certified tank zero.** By design:
  Pico017/Ultra321's baseline is set to match Ultra460's own ambient median, not the
  tank's absolute zero (which would have left Pico017 ~5–6 ppb below Ultra460 — the two
  genuinely disagree on absolute zero). This makes C2H6_cal cross-instrument-consistent
  but **not independently traceable to the certified NOAA/N2-zero tank standards**.
  Downstream analysis should know this before treating C2H6_cal as an absolute measurement.
- **MML-date extrapolation.** All three tank events fell inside the WYO window
  (Feb 3–12). Feb-12 coefficients are applied to the Jan and March MML dates as an
  extrapolation with no direct tank evidence there.
- **C2H6 reference assumption.** The entire C2H6 calibration (baseline and span alike)
  rests on Ultra460's C2H6 being correct, which is assumed, not independently validated.